In [5]:
import pandas as pd

file_path = r"C:\Users\shade\OneDrive\Documents\Samsung Bootcamp\Project\Customer-Support-Tickets\Prompted\data\final_ml_ready_support_tickets.csv"

df = pd.read_csv(file_path)

print(df.shape)

(17217, 19)


In [6]:
sample_ticket = df.iloc[0]

subject = sample_ticket["subject"]
body = sample_ticket["body"]

print("Subject:", subject)
print("\nBody:", body)

Subject: Account Disruption

Body: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?


In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Hello! Reply with: Llama 3.2 is working.",
        "stream": False
    }
)

print(response.json()["response"])

Llama 3 is working.


In [7]:
import joblib

issue_type_tfidf = joblib.load("Models/issue_type_tfidf.pkl")
issue_type_model = joblib.load("Models/issue_type_svm.pkl")

priority_tfidf = joblib.load("Models/priority_tfidf.pkl")
priority_model = joblib.load("Models/priority_svm.pkl")

print("All ML models loaded successfully!")

All ML models loaded successfully!


In [8]:
subject = "Unable to access my account"
body = """
I have been trying to log in to my account since this morning,
but I keep receiving an error message. I need help accessing my account.
"""

ticket_text = subject + " " + body

# Issue Type prediction
issue_type_input = issue_type_tfidf.transform([ticket_text])
predicted_issue_type = issue_type_model.predict(issue_type_input)[0]

# Priority prediction
priority_input = priority_tfidf.transform([ticket_text])
predicted_priority = priority_model.predict(priority_input)[0]

print("Predicted Issue Type:", predicted_issue_type)
print("Predicted Priority:", predicted_priority)

Predicted Issue Type: Incident
Predicted Priority: medium


In [15]:
import requests

prompt = f"""
You are an AI Support Ticket Triage Agent.

You are given a customer support ticket and predictions from two machine learning models.

Ticket Subject:
{subject}

Ticket Body:
{body}

ML Predicted Issue Type:
{predicted_issue_type}

ML Predicted Priority:
{predicted_priority}

Based on the ticket content and the ML predictions, perform the following tasks:

1. Select the most appropriate Queue from ONLY these options:
- Technical Support
- Customer Service
- Billing and Payments
- Product Support
- IT Support
- Returns and Exchanges
- Sales and Pre-Sales
- Human Resources
- Service Outages and Maintenance
- General Inquiry

2. Write a concise Summary of the ticket.
3. Identify the Main Problem.
4. Recommend an appropriate Action for the support team.
5. Write a Suggested Response to the customer.

Important:
- Do not change the ML Predicted Issue Type.
- Do not change the ML Predicted Priority.
- Do not invent specific resolution times, SLAs, deadlines, policies, or guarantees that are not provided in the ticket.
- The Queue must be exactly one of the provided options.
- Keep the response clear and concise.
- Return valid JSON only.
- Use exactly these keys:
  queue
  summary
  main_problem
  recommended_action
  suggested_response
- Do not add any extra text before or after the JSON.
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False
    }
)

llm_result = response.json()["response"]

print(llm_result)

{"queue": "Technical Support", "summary": "Customer unable to access account", "main_problem": "Account access error", "recommended_action": "Escalate to technical team for investigation and resolution", "suggested_response": "We're sorry to hear that you're experiencing issues with your account access. Can you please provide us with more details about the error message you're receiving?"}


In [16]:
import json

# Convert Llama output from JSON string to Python dictionary
triage_result = json.loads(llm_result)

# Display the final triage result
print("=" * 60)
print("AI SUPPORT TICKET TRIAGE RESULT")
print("=" * 60)

print(f"\nIssue Type: {predicted_issue_type}")
print(f"Priority: {predicted_priority}")
print(f"Queue: {triage_result['queue']}")

print(f"\nSummary:")
print(triage_result["summary"])

print(f"\nMain Problem:")
print(triage_result["main_problem"])

print(f"\nRecommended Action:")
print(triage_result["recommended_action"])

print(f"\nSuggested Response:")
print(triage_result["suggested_response"])

AI SUPPORT TICKET TRIAGE RESULT

Issue Type: Incident
Priority: medium
Queue: Technical Support

Summary:
Customer unable to access account

Main Problem:
Account access error

Recommended Action:
Escalate to technical team for investigation and resolution

Suggested Response:
We're sorry to hear that you're experiencing issues with your account access. Can you please provide us with more details about the error message you're receiving?
